# BÀI TẬP LỚN: NỀN TẢNG LẬP TRÌNH CHO PHÂN TÍCH VÀ TRỰC QUAN DỮ LIỆU
**Học kỳ 261 – Năm học 2026–2027** | **Giảng viên:** Lê Thành Sách

---
## BÀI 3: DỮ LIỆU CHUỖI THỜI GIAN (TIME SERIES)
### Đề tài: Dự báo nồng độ bụi mịn PM2.5 trước 24 giờ — So sánh Hà Nội và TP.HCM

- **Nhóm:** [Tên nhóm]
- **Thành viên phụ trách:** [Họ tên – MSSV]

**Bài toán.** Tại thời điểm $t$, dựa vào lịch sử PM2.5 và thời tiết hiện tại, dự báo nồng độ PM2.5
tại thời điểm $t + 24h$ (cùng giờ ngày mai). Đây là bài toán **hồi quy chuỗi thời gian**.

**Vì sao so sánh 2 thành phố?** Hà Nội (4 mùa, mùa đông nghịch nhiệt) và TP.HCM (2 mùa mưa/khô) có cơ chế
ô nhiễm khác nhau. Áp dụng **cùng một pipeline** cho từng thành phố **độc lập** (không gộp dữ liệu) để trả lời:
*mô hình học máy cải thiện bao nhiêu so với baseline ở mỗi nơi, và nơi nào khó dự báo hơn?*

## 0. KHỞI TẠO MÔI TRƯỜNG

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

RANDOM_STATE = 42
HORIZON = 24            # dự báo trước 24 giờ
WHO_24H = 15            # ngưỡng khuyến nghị WHO 2021 cho PM2.5 trung bình 24h (µg/m³)
CITY_COLORS = {'Hà Nội': '#d62728', 'TP.HCM': '#1f77b4'}
print('Môi trường đã sẵn sàng!')

## 1. MÔ TẢ TẬP DỮ LIỆU

### 1.1. Nguồn dữ liệu (tự thu thập)
Dữ liệu được **tự thu thập qua API** bằng script `scripts/crawl_air_quality.py`:

| Thành phần | Nguồn | Ghi chú |
|---|---|---|
| PM2.5 – Hà Nội | [OpenAQ v3](https://openaq.org) – trạm *Số 46 Lưu Quang Vũ* (Hanoi Air Quality Monitoring Network, sensor 7772032) | Trạm quan trắc của thành phố, **có khoảng trống thật** |
| PM2.5 – TP.HCM | OpenAQ v3 – trạm *CMT8* (AirGradient, sensor 11357424) | Cảm biến chi phí thấp |
| Thời tiết | [Open-Meteo Historical Weather API](https://open-meteo.com) | Nhiệt độ, độ ẩm, mưa, gió, áp suất, mây |

- **Giai đoạn:** 01/12/2024 – 30/09/2026 (22 tháng, 16,056 giờ mỗi thành phố), giờ Việt Nam (UTC+7).
- Lưới thời gian chuẩn hóa đầy đủ theo giờ: giờ nào trạm không đo → `NaN`.

| Cột | Kiểu | Ý nghĩa |
|---|---|---|
| `datetime` | datetime | Thời điểm (giờ địa phương) |
| `city` | categorical | Thành phố |
| `temperature_2m` | numerical | Nhiệt độ 2m (°C) |
| `relative_humidity_2m` | numerical | Độ ẩm tương đối (%) |
| `precipitation` | numerical | Lượng mưa (mm) |
| `wind_speed_10m` | numerical | Tốc độ gió 10m (km/h) |
| `wind_direction_10m` | numerical (góc) | Hướng gió (độ) → chuyển thành **categorical** 8 hướng |
| `surface_pressure` | numerical | Áp suất bề mặt (hPa) |
| `cloud_cover` | numerical | Độ che phủ mây (%) |
| `pm25` | numerical | Nồng độ PM2.5 (µg/m³) — **biến cần dự báo** |
| `pm25_source` | categorical | Nguồn PM2.5 (`openaq`, hoặc `open-meteo` nếu crawler phải fallback) |

### 1.2. Nạp dữ liệu
Đọc CSV local; nếu chạy trên Colab không có file thì tải từ GitHub (**sửa `GITHUB_RAW_BASE`** theo repo của nhóm).

In [ ]:
GITHUB_RAW_BASE = 'https://raw.githubusercontent.com/perckgg/NTLT/main/03_TimeSeries/data/timeseries'
CITY_FILES = {'Hà Nội': 'air_quality_hanoi.csv', 'TP.HCM': 'air_quality_hcmc.csv'}
LOCAL_DIRS = ['data/timeseries', '../data/timeseries', '/content/data/timeseries', '/content']


def load_city_data(filename):
    for d in LOCAL_DIRS:
        path = os.path.join(d, filename)
        if os.path.exists(path):
            return pd.read_csv(path, parse_dates=['datetime'])
    return pd.read_csv(f'{GITHUB_RAW_BASE}/{filename}', parse_dates=['datetime'])


raw = {city: load_city_data(f).set_index('datetime').sort_index() for city, f in CITY_FILES.items()}
for city, df in raw.items():
    print(f'{city}: {df.shape[0]:,} dòng × {df.shape[1]} cột | '
          f'{df.index.min():%Y-%m-%d} → {df.index.max():%Y-%m-%d} | nguồn PM2.5: {", ".join(df.pm25_source.unique())}')
raw['Hà Nội'].head()

### 1.3. Thống kê mô tả

In [ ]:
raw['Hà Nội'].info()
summary = pd.concat({city: df['pm25'].describe() for city, df in raw.items()}, axis=1).T
summary['missing_rate'] = [df['pm25'].isna().mean() for df in raw.values()]
summary['% giờ > WHO'] = [(df['pm25'] > WHO_24H).mean() for df in raw.values()]
display(summary.round(3))
display(pd.concat({city: df.describe().T for city, df in raw.items()}).round(2))

### 1.4. Trực quan hóa chuỗi thời gian
**Hình 1 – PM2.5 trung bình ngày** của hai thành phố so với ngưỡng WHO 24h (15 µg/m³).

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
for city, df in raw.items():
    daily = df['pm25'].resample('D').mean()
    ax.plot(daily.index, daily, label=city, color=CITY_COLORS[city], lw=0.9)
ax.axhline(WHO_24H, color='k', ls='--', lw=1, label='Ngưỡng WHO 24h (15 µg/m³)')
ax.set(title='PM2.5 trung bình ngày', ylabel='PM2.5 (µg/m³)')
ax.legend()
plt.show()

**Hình 2 – Phân phối PM2.5 theo giờ** và **Hình 3 – Boxplot theo tháng** (tính mùa vụ).

In [ ]:
plot_df = pd.concat([df[['pm25']].assign(city=city) for city, df in raw.items()])
plot_df['month'] = plot_df.index.month
plot_df['hour'] = plot_df.index.hour
plot_df = plot_df.reset_index(drop=True)

fig, ax = plt.subplots(figsize=(12, 4))
sns.histplot(data=plot_df, x='pm25', hue='city', bins=100, element='step', stat='density',
             common_norm=False, palette=CITY_COLORS, ax=ax)
ax.set(title='Phân phối PM2.5 theo giờ', xlabel='PM2.5 (µg/m³)', xlim=(0, 250))
plt.show()

fig, ax = plt.subplots(figsize=(14, 5))
sns.boxplot(data=plot_df, x='month', y='pm25', hue='city', palette=CITY_COLORS, showfliers=False, ax=ax)
ax.set(title='PM2.5 theo tháng (ẩn outlier để dễ nhìn)', xlabel='Tháng', ylabel='PM2.5 (µg/m³)')
plt.show()

**Hình 4 – Chu kỳ ngày:** PM2.5 trung bình theo giờ, tách nửa năm khô/lạnh (tháng 11–4) và mưa/nóng (tháng 5–10).

In [ ]:
plot_df['half_year'] = np.where(plot_df['month'].isin([11, 12, 1, 2, 3, 4]), 'Tháng 11–4', 'Tháng 5–10')
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, city in zip(axes, CITY_FILES):
    g = plot_df[plot_df.city == city]
    sns.lineplot(data=g, x='hour', y='pm25', hue='half_year', errorbar=None, ax=ax)
    ax.set(title=city, xlabel='Giờ trong ngày', ylabel='PM2.5 trung bình (µg/m³)', xticks=range(0, 24, 3))
plt.tight_layout()
plt.show()

**Hình 5 – Ma trận tương quan (Spearman)** giữa PM2.5 và các biến thời tiết, từng thành phố.

In [ ]:
num_cols = ['pm25', 'temperature_2m', 'relative_humidity_2m', 'precipitation',
            'wind_speed_10m', 'surface_pressure', 'cloud_cover']
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, (city, df) in zip(axes, raw.items()):
    sns.heatmap(df[num_cols].corr(method='spearman'), annot=True, fmt='.2f', cmap='RdBu_r',
                vmin=-1, vmax=1, ax=ax, cbar=False)
    ax.set_title(f'Tương quan Spearman – {city}')
plt.tight_layout()
plt.show()

**Nhận xét EDA:**


- **Mức ô nhiễm trung bình tương đương nhưng có hình dạng khác nhau:** PM2.5 trung bình Hà Nội 31.7 µg/m³, TP.HCM 32.1 µg/m³.
  Hà Nội biến động mạnh hơn (std 20.6 so với 16.3; max 332 so với 171). Cả hai đều vượt ngưỡng WHO 24h (15 µg/m³)
  trong phần lớn số giờ: Hà Nội 56%, TP.HCM 82%.
- **Theo mùa:** Hà Nội có mùa ô nhiễm rõ rệt vào mùa đông (tháng 12: 53, tháng 1: 55 µg/m³) và sạch nhất vào tháng 8 (19 µg/m³),
  tức chênh lệch gần 3 lần. TP.HCM cũng ô nhiễm cao nhất vào cuối mùa khô / đầu năm (tháng 1: 46 µg/m³) và thấp nhất vào giữa mùa mưa (tháng 7: 24 µg/m³), nhưng biên độ hẹp hơn.
- **Theo giờ:** TP.HCM có đỉnh buổi sáng lúc 7h (≈41 µg/m³), trùng giờ cao điểm giao thông; Hà Nội đỉnh vào buổi tối (≈34 µg/m³ lúc 22h),
  phù hợp với việc lớp nghịch nhiệt ban đêm giữ bụi lại gần mặt đất.
- **Tương quan với thời tiết:** Hà Nội tương quan âm với nhiệt độ (-0.31), mây (-0.25) và dương với áp suất (+0.31), cho thấy
  không khí lạnh, áp cao của mùa đông làm tích tụ bụi. TP.HCM phụ thuộc nhiều nhất vào tốc độ gió (-0.36): gió mạnh thì bụi được phát tán.
- **Tự tương quan:** tự tương quan lag 1h rất cao (0.92 / 0.88), còn lag 24h giảm xuống 0.56 / 0.45. Vì vậy dự báo trước 24h
  khó hơn đáng kể so với dự báo giờ kế tiếp, và cần thêm đặc trưng thời tiết, mùa, chu kỳ ngày.
- **Chất lượng dữ liệu:** Hà Nội thiếu 32.8% giờ PM2.5, tập trung từ cuối 2025 (tháng 12/2025 thiếu 97%); TP.HCM thiếu 8.4%.
  Đây là vấn đề phải xử lý kỹ ở Phần 2.

## 2. CHUẨN BỊ DỮ LIỆU

Các bước dưới đây được viết thành **hàm tái sử dụng** và áp dụng **giống hệt nhau** cho từng thành phố,
nhưng mỗi thành phố được xử lý **độc lập**.

### 2.1. Thống kê missing values

In [ ]:
def gap_lengths(series):
    """Độ dài (giờ) của từng đoạn missing liên tiếp."""
    is_na = series.isna()
    groups = (is_na != is_na.shift()).cumsum()
    return is_na.groupby(groups).sum()[lambda s: s > 0]


def missing_report(raw_dict):
    rows = []
    for city, df in raw_dict.items():
        gaps = gap_lengths(df['pm25'])
        rows.append({'Thành phố': city, 'Số giờ thiếu': int(df['pm25'].isna().sum()),
                     'Tỉ lệ thiếu': f"{df['pm25'].isna().mean():.1%}",
                     'Số đoạn thiếu': len(gaps), 'Đoạn ≤ 6h': int((gaps <= 6).sum()),
                     'Đoạn dài nhất (giờ)': int(gaps.max()) if len(gaps) else 0})
    return pd.DataFrame(rows)


display(missing_report(raw))
display((pd.concat({city: df.isna().mean() for city, df in raw.items()}, axis=1) * 100)
        .round(2).rename(columns=lambda c: f'{c} (% missing)'))

fig, ax = plt.subplots(figsize=(14, 4))
for city, df in raw.items():
    avail = df['pm25'].notna().resample('MS').mean() * 100
    ax.plot(avail.index, avail, marker='o', label=city, color=CITY_COLORS[city])
ax.set(title='Tỉ lệ giờ có dữ liệu PM2.5 theo tháng', ylabel='% giờ có dữ liệu', ylim=(0, 105))
ax.legend()
plt.show()

### 2.2. Làm sạch giá trị lỗi và xử lý missing

- **Giá trị không hợp lệ:** PM2.5 âm hoặc > 1000 µg/m³ là lỗi cảm biến → gán `NaN`.
- **Cảm biến bị "kẹt":** cùng một giá trị lặp lại ≥ 6 giờ liên tiếp (khi PM2.5 > 0) là dấu hiệu treo thiết bị → gán `NaN`.
- **Khoảng trống ngắn (≤ 6 giờ):** nội suy tuyến tính theo thời gian (`interpolate(method='time', limit=6)`) —
  hợp lý vì PM2.5 biến đổi liên tục trong vài giờ.
- **Khoảng trống dài (> 6 giờ):** **không** nội suy (dễ bịa dữ liệu) → mẫu có *target* bị thiếu sẽ bị loại;
  feature còn thiếu được `SimpleImputer` (median, học trên tập train) xử lý bên trong pipeline.

In [ ]:
MAX_INTERP_GAP = 6
STUCK_HOURS = 6


def clean_city(df):
    df = df.copy()
    pm = df['pm25']
    invalid = (pm < 0) | (pm > 1000)
    run_id = (pm != pm.shift()).cumsum()
    run_len = pm.groupby(run_id).transform('size')
    stuck = (run_len >= STUCK_HOURS) & (pm > 0) & pm.notna()
    df.loc[invalid | stuck, 'pm25'] = np.nan
    df['pm25_raw'] = df['pm25']
    df['pm25'] = df['pm25'].interpolate(method='time', limit=MAX_INTERP_GAP, limit_area='inside')
    return df, int(invalid.sum()), int(stuck.sum())


clean = {}
for city, df in raw.items():
    clean[city], n_invalid, n_stuck = clean_city(df)
    before, after = df['pm25'].isna().mean(), clean[city]['pm25'].isna().mean()
    print(f'{city}: {n_invalid} giá trị âm/>1000, {n_stuck} giờ cảm biến kẹt | '
          f'missing PM2.5: {before:.1%} → {after:.1%} sau làm sạch + nội suy')


def first_short_gap(series, min_len=3, max_len=MAX_INTERP_GAP):
    is_na = series.isna()
    groups = (is_na != is_na.shift()).cumsum()
    sizes = is_na.groupby(groups).transform('sum')
    mask = is_na & (sizes >= min_len) & (sizes <= max_len)
    return series.index[mask][0] if mask.any() else None


for city in clean:
    gap_time = first_short_gap(clean[city]['pm25_raw'])
    if gap_time is not None:
        break
if gap_time is not None:
    window = clean[city].loc[gap_time - pd.Timedelta('36h'): gap_time + pd.Timedelta('36h')]
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(window.index, window['pm25'], 'o--', color='orange', label='Sau nội suy')
    ax.plot(window.index, window['pm25_raw'], 'o-', color=CITY_COLORS[city], label='Dữ liệu gốc')
    ax.set(title=f'Minh họa nội suy khoảng trống ngắn – {city}', ylabel='PM2.5 (µg/m³)')
    ax.legend()
    plt.show()

### 2.3. Phát hiện outliers (IQR)
Quy tắc IQR: giá trị ngoài $[Q1 - 3\cdot IQR,\ Q3 + 3\cdot IQR]$ được coi là outlier cực trị.

In [ ]:
def iqr_bounds(s, k=3.0):
    q1, q3 = s.quantile([0.25, 0.75])
    return q1 - k * (q3 - q1), q3 + k * (q3 - q1)


fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (city, df) in zip(axes, clean.items()):
    lo, hi = iqr_bounds(df['pm25'])
    out = df.loc[df['pm25'] > hi, 'pm25']
    top_days = out.resample('D').count().nlargest(3).index.strftime('%Y-%m-%d').tolist()
    print(f'{city}: ngưỡng trên = {hi:.1f} µg/m³ | {len(out)} giờ outlier '
          f'({len(out) / df.pm25.notna().sum():.2%}) | ngày nhiều outlier nhất: {top_days}')
    ax.plot(df.index, df['pm25'], lw=0.4, color=CITY_COLORS[city])
    ax.scatter(out.index, out, s=6, color='k', label='Outlier (IQR, k=3)')
    ax.axhline(hi, color='k', ls='--', lw=1)
    ax.set(title=city, ylabel='PM2.5 (µg/m³)')
    ax.legend()
plt.tight_layout()
plt.show()

**Quyết định xử lý outlier:** các đỉnh PM2.5 là **sự kiện có thật** (đợt ô nhiễm mùa đông, đốt rơm rạ, pháo hoa...)
nên **không xóa** khỏi target (xóa sẽ khiến đánh giá lạc quan giả tạo). Thay vào đó, trên các **feature** PM2.5
(lag, rolling) dùng transformer tự viết `IQRClipper` (học ngưỡng **chỉ trên tập train**) để giảm ảnh hưởng của
giá trị cực trị tới mô hình tuyến tính.

### 2.4. Tạo đặc trưng (Feature Engineering)

Mỗi mẫu tại thời điểm $t$ có target `y = pm25(t + 24h)`. Chỉ dùng thông tin **có sẵn tại $t$** (tránh rò rỉ tương lai):

| Nhóm | Feature | Loại |
|---|---|---|
| Lịch sử PM2.5 | `pm25_lag_0, 1, 2, 3, 6, 12, 23` (giá trị tại $t-k$), `pm25_roll_mean_6/24`, `pm25_roll_std_24`, `pm25_diff_24` | numerical |
| Thời tiết tại $t$ | nhiệt độ, độ ẩm, mưa, tốc độ gió, áp suất, mây, `rain_24h` (tổng mưa 24h qua) | numerical |
| Lịch của thời điểm cần dự báo | `hour_sin/cos` (mã hóa vòng tròn giờ) | numerical |
| | `season` (mùa theo khí hậu từng thành phố), `day_of_week`, `wind_sector` (8 hướng gió) | **categorical** |

> `pm25_lag_0` là giá trị **cùng giờ ngày hôm trước** so với thời điểm cần dự báo, nên baseline
> **Persistence** (`ŷ(t+24) = pm25(t)`) cũng chính là baseline *seasonal naive* 24h.

In [ ]:
SEASONS = {
    'Hà Nội': {12: 'Đông', 1: 'Đông', 2: 'Đông', 3: 'Xuân', 4: 'Xuân', 5: 'Xuân',
               6: 'Hạ', 7: 'Hạ', 8: 'Hạ', 9: 'Thu', 10: 'Thu', 11: 'Thu'},
    'TP.HCM': {m: ('Mùa khô' if m in (12, 1, 2, 3, 4) else 'Mùa mưa') for m in range(1, 13)},
}
WIND_SECTORS = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']
LAGS = [0, 1, 2, 3, 6, 12, 23]
WEATHER_RAW = ['temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_speed_10m',
               'surface_pressure', 'cloud_cover']
WEATHER_FEATURES = WEATHER_RAW + ['rain_24h']
PM_FEATURES = [f'pm25_lag_{k}' for k in LAGS] + ['pm25_roll_mean_6', 'pm25_roll_mean_24',
                                                 'pm25_roll_std_24', 'pm25_diff_24']
CYCLIC_FEATURES = ['hour_sin', 'hour_cos']
CAT_FEATURES = ['season', 'day_of_week', 'wind_sector']
FEATURES = PM_FEATURES + WEATHER_FEATURES + CYCLIC_FEATURES + CAT_FEATURES
TARGET = 'target'


def build_features(df, city, horizon=HORIZON):
    """Tạo feature tại mỗi thời điểm t (chỉ dùng dữ liệu ≤ t)."""
    pm = df['pm25']
    X = pd.DataFrame(index=df.index)
    for k in LAGS:
        X[f'pm25_lag_{k}'] = pm.shift(k)
    X['pm25_roll_mean_6'] = pm.rolling(6, min_periods=3).mean()
    X['pm25_roll_mean_24'] = pm.rolling(24, min_periods=12).mean()
    X['pm25_roll_std_24'] = pm.rolling(24, min_periods=12).std()
    X['pm25_diff_24'] = pm - pm.shift(24)
    X[WEATHER_RAW] = df[WEATHER_RAW]
    X['rain_24h'] = df['precipitation'].rolling(24, min_periods=1).sum()

    target_time = df.index + pd.Timedelta(hours=horizon)
    X['hour_sin'] = np.sin(2 * np.pi * target_time.hour / 24)
    X['hour_cos'] = np.cos(2 * np.pi * target_time.hour / 24)
    X['season'] = pd.Series(target_time.month, index=df.index).map(SEASONS[city])
    X['day_of_week'] = target_time.day_name()
    sector_idx = ((df['wind_direction_10m'] + 22.5) % 360) // 45
    X['wind_sector'] = sector_idx.map(lambda i: WIND_SECTORS[int(i)] if pd.notna(i) else np.nan)
    return X


def make_dataset(df, city, horizon=HORIZON):
    """Thêm target = pm25(t + horizon); chỉ giữ mẫu có target và có giá trị hiện tại."""
    X = build_features(df, city, horizon)
    X[TARGET] = df['pm25'].shift(-horizon)
    return X.dropna(subset=[TARGET, 'pm25_lag_0'])


datasets = {city: make_dataset(df, city) for city, df in clean.items()}
for city, d in datasets.items():
    print(f'{city}: {len(d):,} mẫu | {len(FEATURES)} feature '
          f'({len(FEATURES) - len(CAT_FEATURES)} numerical + {len(CAT_FEATURES)} categorical)')
datasets['Hà Nội'].head()

### 2.5. Chia tập Train / Validation / Test theo thời gian
Chuỗi thời gian **không được xáo trộn ngẫu nhiên** (sẽ rò rỉ tương lai vào quá khứ). Chia **theo thứ tự thời gian**:
70% đầu → Train, 15% kế tiếp → Validation (chọn mô hình), 15% cuối → Test (đánh giá cuối cùng).
Bỏ thêm `HORIZON` = 24 giờ trước mỗi ranh giới để **target** của tập trước không rơi vào khoảng thời gian của tập sau.

In [ ]:
def time_split(d, train_frac=0.70, val_frac=0.15, gap_hours=HORIZON):
    n = len(d)
    t1, t2 = d.index[int(n * train_frac)], d.index[int(n * (train_frac + val_frac))]
    gap = pd.Timedelta(hours=gap_hours)
    train = d[d.index < t1 - gap]
    val = d[(d.index >= t1) & (d.index < t2 - gap)]
    test = d[d.index >= t2]
    return train, val, test


splits = {city: time_split(d) for city, d in datasets.items()}

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
for ax, (city, (tr, va, te)) in zip(axes, splits.items()):
    for part, name, color in [(tr, 'Train', 'tab:blue'), (va, 'Validation', 'tab:orange'), (te, 'Test', 'tab:green')]:
        daily = part[TARGET].resample('D').mean()
        ax.plot(daily.index, daily, color=color, lw=0.8,
                label=f'{name}: {len(part):,} mẫu ({part.index.min():%d/%m/%Y} – {part.index.max():%d/%m/%Y})')
    ax.set(title=city, ylabel='PM2.5 (µg/m³)')
    ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()

### 2.6. Pipeline tiền xử lý (Imputation, Outlier clipping, Encoding, Scaling)

| Nhóm cột | Các bước |
|---|---|
| PM2.5 lag/rolling | `SimpleImputer(median)` → `IQRClipper(k=3)` → `StandardScaler` |
| Thời tiết + chu kỳ giờ | `SimpleImputer(median)` → `StandardScaler` |
| Categorical | `SimpleImputer(most_frequent)` → `OneHotEncoder` |

Tất cả nằm trong `ColumnTransformer` + `Pipeline` nên các tham số (median, ngưỡng IQR, mean/std, danh mục)
**chỉ được học trên tập train** → không rò rỉ dữ liệu.

In [ ]:
class IQRClipper(BaseEstimator, TransformerMixin):
    """Cắt (winsorize) mỗi cột về [Q1 - k·IQR, Q3 + k·IQR], ngưỡng học trên dữ liệu fit."""

    def __init__(self, k=3.0):
        self.k = k

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        q1, q3 = np.nanpercentile(X, [25, 75], axis=0)
        self.lower_, self.upper_ = q1 - self.k * (q3 - q1), q3 + self.k * (q3 - q1)
        return self

    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.lower_, self.upper_)

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)


def build_preprocessor():
    return ColumnTransformer([
        ('pm', Pipeline([('impute', SimpleImputer(strategy='median')),
                         ('clip', IQRClipper(k=3.0)),
                         ('scale', StandardScaler())]), PM_FEATURES),
        ('num', Pipeline([('impute', SimpleImputer(strategy='median')),
                          ('scale', StandardScaler())]), WEATHER_FEATURES + CYCLIC_FEATURES),
        ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                          ('onehot', OneHotEncoder(handle_unknown='ignore'))]), CAT_FEATURES),
    ], sparse_threshold=0)


tr = splits['Hà Nội'][0]
pre = build_preprocessor().fit(tr[FEATURES])
Xt = pd.DataFrame(pre.transform(tr[FEATURES]), columns=pre.get_feature_names_out(), index=tr.index)
print(f'Sau tiền xử lý: {Xt.shape[1]} cột (từ {len(FEATURES)} cột gốc, one-hot mở rộng categorical)')
print('Missing còn lại sau pipeline:', int(Xt.isna().sum().sum()))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(tr['pm25_lag_0'], bins=80, ax=axes[0], color='gray')
axes[0].set(title='pm25_lag_0 – trước xử lý (Hà Nội, train)', xlabel='µg/m³')
sns.histplot(Xt['pm__pm25_lag_0'], bins=80, ax=axes[1], color='teal')
axes[1].set(title='Sau IQRClipper + StandardScaler', xlabel='z-score')
plt.tight_layout()
plt.show()

## 3. PHÂN TÍCH VÀ MÔ HÌNH HÓA

### 3.1. Các mô hình so sánh
| Mô hình | Vai trò | Lý do chọn |
|---|---|---|
| **Persistence (baseline)** | `ŷ(t+24) = pm25(t)` | Baseline chuẩn cho chuỗi thời gian: mô hình ML phải **thắng** được nó mới có giá trị |
| **Ridge Regression** | Tuyến tính có regularization | Đơn giản, dễ giải thích; kiểm tra mức độ quan hệ tuyến tính |
| **Random Forest** | Ensemble cây (bagging) | Bắt được quan hệ phi tuyến, tương tác thời tiết × giờ |
| **HistGradientBoosting** | Ensemble cây (boosting) | Mạnh và nhanh trên dữ liệu bảng lớn |

**Tối ưu siêu tham số:** `GridSearchCV` với `TimeSeriesSplit(n_splits=3)` trên tập train (fold validation luôn nằm
**sau** fold train), tiêu chí `MAE`. Random Forest dùng cấu hình cố định (giới hạn `max_depth`, `min_samples_leaf`
để chống overfit) nhằm giữ thời gian chạy hợp lý trên Colab.

Lớp `CityForecaster` đóng gói toàn bộ quy trình cho **một** thành phố, để chạy lặp lại cho cả hai.

In [ ]:
class PersistenceBaseline(BaseEstimator, RegressorMixin):
    """Dự báo PM2.5 sau HORIZON giờ bằng giá trị hiện tại."""

    def fit(self, X, y=None):
        return self

    def predict(self, X):
        return np.asarray(X['pm25_lag_0'], dtype=float)


def regression_metrics(y_true, y_pred):
    return {'MAE': mean_absolute_error(y_true, y_pred),
            'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
            'R2': r2_score(y_true, y_pred)}


def make_models():
    return {
        'Persistence (baseline)': PersistenceBaseline(),
        'Ridge': Pipeline([('pre', build_preprocessor()), ('model', Ridge())]),
        'RandomForest': Pipeline([('pre', build_preprocessor()), ('model', RandomForestRegressor(
            n_estimators=200, max_depth=12, min_samples_leaf=10, max_features=0.5,
            n_jobs=-1, random_state=RANDOM_STATE))]),
        'HistGradientBoosting': Pipeline([('pre', build_preprocessor()),
                                          ('model', HistGradientBoostingRegressor(random_state=RANDOM_STATE))]),
    }


PARAM_GRIDS = {
    'Ridge': {'model__alpha': [0.1, 1, 10, 100]},
    'HistGradientBoosting': {'model__learning_rate': [0.03, 0.1],
                             'model__max_iter': [200, 400],
                             'model__max_leaf_nodes': [15, 31],
                             'model__l2_regularization': [0.0, 1.0]},
}


class CityForecaster:
    def __init__(self, city, train, val, test):
        self.city, self.train, self.val, self.test = city, train, val, test
        self.models = make_models()
        self.best_params = {}

    def tune(self):
        for name, grid in PARAM_GRIDS.items():
            search = GridSearchCV(self.models[name], grid, cv=TimeSeriesSplit(n_splits=3),
                                  scoring='neg_mean_absolute_error', n_jobs=-1)
            search.fit(self.train[FEATURES], self.train[TARGET])
            self.models[name] = search.best_estimator_
            self.best_params[name] = search.best_params_
        return self

    def evaluate_on_validation(self):
        rows = []
        for name, m in self.models.items():
            m.fit(self.train[FEATURES], self.train[TARGET])
            rows.append({'city': self.city, 'model': name,
                         **regression_metrics(self.val[TARGET], m.predict(self.val[FEATURES]))})
        return pd.DataFrame(rows)

    def evaluate_on_test(self):
        """Huấn luyện lại trên Train + Validation rồi đánh giá một lần trên Test."""
        full = pd.concat([self.train, self.val])
        rows, self.test_predictions = [], pd.DataFrame({'actual': self.test[TARGET]})
        for name, m in self.models.items():
            m.fit(full[FEATURES], full[TARGET])
            pred = m.predict(self.test[FEATURES])
            self.test_predictions[name] = pred
            rows.append({'city': self.city, 'model': name, **regression_metrics(self.test[TARGET], pred)})
        return pd.DataFrame(rows)


forecasters = {city: CityForecaster(city, *parts).tune() for city, parts in splits.items()}
for city, f in forecasters.items():
    print(f'{city} – siêu tham số tốt nhất:')
    for name, params in f.best_params.items():
        print(f'   {name}: {params}')

### 3.2. So sánh trên tập Validation

In [ ]:
val_results = pd.concat([f.evaluate_on_validation() for f in forecasters.values()], ignore_index=True)
display(val_results.pivot(index='model', columns='city', values=['MAE', 'RMSE', 'R2']).round(3))

**Nhận xét Validation:**
- Ở cả hai thành phố, mọi mô hình học máy đều có MAE thấp hơn baseline Persistence. Mức cải thiện lớn ở Hà Nội
  (14.97 → 12.38 với RandomForest, giảm ≈17%) và nhỏ ở TP.HCM (7.71 → 7.07 với Ridge, giảm ≈8%).
- Baseline có R² âm (-0.06 / -0.15): lấy nguyên giá trị hiện tại làm dự báo cho 24h sau còn tệ hơn dự báo bằng giá trị trung bình.
- Ở Hà Nội, ba mô hình học máy chênh nhau rất ít (MAE 12.4–12.6). Ở TP.HCM, Ridge tốt nhất: khi tín hiệu yếu (R² chỉ 0.08–0.14),
  mô hình tuyến tính có regularization mạnh (alpha=100) ít bị overfit hơn mô hình cây.
- GridSearch chọn learning_rate nhỏ (0.03) và cây ít lá (15) cho HGB, tức là ưu tiên mô hình đơn giản. Điều này nhất quán với
  việc dữ liệu nhiễu và chỉ dài 22 tháng.

## 4. TỔNG KẾT VÀ MỞ RỘNG

### 4.1. Kết quả trên tập Test độc lập
Mỗi mô hình được huấn luyện lại trên Train + Validation (siêu tham số đã cố định), rồi đánh giá **một lần** trên Test.

In [ ]:
test_results = pd.concat([f.evaluate_on_test() for f in forecasters.values()], ignore_index=True)
base_mae = test_results[test_results.model == 'Persistence (baseline)'].set_index('city')['MAE']
test_results['MAE giảm so với baseline'] = test_results.apply(lambda r: 1 - r['MAE'] / base_mae[r['city']], axis=1)
display(test_results.set_index(['city', 'model']).style.format(
    {'MAE': '{:.2f}', 'RMSE': '{:.2f}', 'R2': '{:.3f}', 'MAE giảm so với baseline': '{:+.1%}'}))
print('Test:', {c: f.test.index.min().strftime('%Y-%m-%d') + ' → ' + f.test.index.max().strftime('%Y-%m-%d')
                for c, f in forecasters.items()})

### 4.2. Câu hỏi: *Mô hình học máy tốt hơn baseline bao nhiêu ở mỗi thành phố?*
**Hình 6 – MAE trên tập Test** của từng mô hình, tách theo thành phố (nhãn trên cột: % MAE giảm so với baseline Persistence).

In [ ]:
MODEL_ORDER = list(make_models())
fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=test_results, x='model', y='MAE', hue='city', order=MODEL_ORDER,
            hue_order=list(CITY_FILES), palette=CITY_COLORS, ax=ax)
for container, city in zip(ax.containers, CITY_FILES):
    sub = test_results[test_results.city == city].set_index('model')
    ax.bar_label(container, labels=[f"{sub.loc[m, 'MAE giảm so với baseline']:+.0%}" for m in MODEL_ORDER],
                 fontsize=9, padding=2)
ax.set(title='MAE trên tập Test (thấp hơn là tốt hơn)', xlabel='', ylabel='MAE (µg/m³)')
plt.show()

**Chú thích Hình 6:** Trên tập Test độc lập, mô hình tốt nhất giảm MAE so với baseline Persistence
**13.9% ở Hà Nội** (RandomForest: 8.32 → 7.16 µg/m³) và **10.6% ở TP.HCM** (Ridge: 9.53 → 8.52 µg/m³).
Mô hình tốt nhất khác nhau giữa hai thành phố: Hà Nội hưởng lợi từ mô hình phi tuyến (RandomForest, HGB) nhờ tương tác
giữa mùa, mưa và mức bụi gần đây, còn ở TP.HCM mô hình tuyến tính Ridge lại ổn định nhất.
Như vậy **không có một mô hình chung tốt nhất cho cả hai thành phố**, và đó là lý do nên huấn luyện riêng từng nơi thay vì gộp dữ liệu.
Lưu ý: hai tập Test không cùng giai đoạn (Hà Nội 19/03–29/09/2026, TP.HCM 29/06–29/09/2026, do Hà Nội thiếu nhiều dữ liệu hơn)
và đều rơi vào giai đoạn ít ô nhiễm. Vì vậy không nên so sánh trực tiếp giá trị MAE tuyệt đối giữa hai thành phố, mà nên so sánh mức cải thiện tương đối so với baseline.
(MAE Test của Hà Nội thấp hơn Validation (7.2 so với 12.4) vì giai đoạn Test không có mùa đông; ở TP.HCM thì ngược lại (8.5 so với 7.1) do mức bụi tăng trở lại vào tháng 9.)

### 4.3. Dự báo so với thực tế
**Hình 7 – 14 ngày đầu của tập Test:** giá trị thực, baseline và mô hình tốt nhất của từng thành phố
(trục thời gian là thời điểm **được dự báo**, tức $t + 24h$).

In [ ]:
best_model = test_results.loc[test_results.groupby('city')['MAE'].idxmin()].set_index('city')['model']

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
for ax, (city, f) in zip(axes, forecasters.items()):
    p = f.test_predictions
    w = p.loc[: p.index.min() + pd.Timedelta(days=14)]
    target_time = w.index + pd.Timedelta(hours=HORIZON)
    ax.plot(target_time, w['actual'], color='k', lw=1.3, label='Thực tế')
    ax.plot(target_time, w['Persistence (baseline)'], color='gray', ls=':', label='Persistence (baseline)')
    ax.plot(target_time, w[best_model[city]], color=CITY_COLORS[city], label=best_model[city])
    ax.set(title=f'{city} – dự báo trước 24h', ylabel='PM2.5 (µg/m³)')
    ax.legend()
plt.tight_layout()
plt.show()

### 4.4. Phân tích lỗi (Error Analysis)
Bài toán hồi quy không có confusion matrix trực tiếp, nên ta (1) phân tích sai số theo **giờ trong ngày** và **tháng**,
(2) quy đổi PM2.5 về **mức chất lượng không khí** (ngưỡng US EPA 2024) để lập **confusion matrix theo mức** —
cho biết mô hình hay đoán nhầm mức nào sang mức nào.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for city, f in forecasters.items():
    p = f.test_predictions
    err = (p[best_model[city]] - p['actual']).abs()
    t = p.index + pd.Timedelta(hours=HORIZON)
    err.groupby(t.hour).mean().plot(ax=axes[0], marker='o', color=CITY_COLORS[city], label=city)
    by_hour, by_month = err.groupby(t.hour).mean(), err.groupby(t.strftime('%Y-%m')).mean()
    by_month.plot(ax=axes[1], marker='o', color=CITY_COLORS[city], label=city)
    print(f'{city}: MAE theo giờ min {by_hour.min():.1f} (h{by_hour.idxmin()}) – max {by_hour.max():.1f} '
          f'(h{by_hour.idxmax()}) | theo tháng: {by_month.round(1).to_dict()}')
axes[0].set(title='MAE theo giờ trong ngày (mô hình tốt nhất)', xlabel='Giờ', ylabel='MAE (µg/m³)')
axes[1].set(title='MAE theo tháng của tập Test (mô hình tốt nhất)', xlabel='Tháng', ylabel='MAE (µg/m³)')
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
AQI_BINS = [-np.inf, 9.0, 35.4, 55.4, 125.4, np.inf]
AQI_LABELS = ['Tốt', 'Trung bình', 'Kém (nhạy cảm)', 'Xấu', 'Rất xấu+']


def to_aqi_level(values):
    return pd.cut(pd.Series(np.asarray(values, dtype=float)), bins=AQI_BINS, labels=AQI_LABELS)


fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, (city, f) in zip(axes, forecasters.items()):
    p = f.test_predictions
    actual_lvl, pred_lvl = to_aqi_level(p['actual']), to_aqi_level(p[best_model[city]])
    cm = pd.crosstab(actual_lvl, pred_lvl, rownames=['Thực tế'], colnames=['Dự báo'], dropna=False)
    cm = cm.reindex(index=AQI_LABELS, columns=AQI_LABELS, fill_value=0)
    cm_norm = cm.div(cm.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
    sns.heatmap(cm_norm, annot=cm.values, fmt='d', cmap='Blues', cbar=False, ax=ax)
    acc = (actual_lvl == pred_lvl).mean()
    ax.set_title(f'{city} – {best_model[city]} | đúng mức: {acc:.1%}')
    print(f'{city}: dự báo đúng mức AQI {acc:.1%} | đúng ±1 mức: '
          f'{(abs(actual_lvl.cat.codes - pred_lvl.cat.codes) <= 1).mean():.1%}')
    print(cm, end='\n\n')
plt.tight_layout()
plt.show()

**Nhận xét phân tích lỗi:**
- **Theo giờ:** sai số Hà Nội khá đều trong ngày (6.2–8.5 µg/m³). TP.HCM sai nhiều nhất vào ban ngày (đỉnh 9.9 lúc 14h),
  là lúc giao thông và đối lưu làm PM2.5 dao động mạnh.
- **Theo tháng:** sai số tăng dần về cuối giai đoạn Test (TP.HCM: 3.2 vào cuối tháng 6 → 10.9 vào tháng 9/2026), khi mức ô nhiễm bắt đầu tăng trở lại.
  Mô hình học chủ yếu trên phân phối cũ nên phản ứng chậm với thay đổi theo mùa.
- **Theo mức AQI:** dự báo đúng mức 82.6% (Hà Nội) và 76.4% (TP.HCM); lệch không quá 1 mức là 99.1% / 98.6%.
  Lỗi chủ yếu nằm ở việc **dự báo thấp các giờ ô nhiễm**: ở TP.HCM, 281/359 giờ "Kém" bị đoán thành "Trung bình", và 29/71 giờ "Xấu"
  bị đoán thành "Trung bình". Mô hình không bao giờ dự báo mức "Tốt", tức là có xu hướng kéo dự báo về giá trị trung bình.
- **Hệ quả thực tế:** mô hình hữu ích để nắm xu hướng chung, nhưng **chưa đủ tin cậy để cảnh báo sớm các đợt ô nhiễm nặng**.
  Đây là trường hợp mà một lỗi bỏ sót gây hậu quả lớn hơn một lỗi báo nhầm.

### 4.5. Mức độ quan trọng của đặc trưng (Permutation Importance)
Đo mức tăng MAE trên tập Test khi xáo trộn ngẫu nhiên từng feature gốc của mô hình tốt nhất.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, (city, f) in zip(axes, forecasters.items()):
    model = f.models[best_model[city]]
    imp = permutation_importance(model, f.test[FEATURES], f.test[TARGET], n_repeats=5,
                                 scoring='neg_mean_absolute_error', random_state=RANDOM_STATE)
    s = pd.Series(imp.importances_mean, index=FEATURES).nlargest(10).sort_values()
    s.plot.barh(ax=ax, color=CITY_COLORS[city])
    print(f'{city} – top 5:', s[::-1].head(5).round(2).to_dict())
    ax.set(title=f'{city} – Top 10 feature ({best_model[city]})', xlabel='Mức tăng MAE (µg/m³)')
plt.tight_layout()
plt.show()

**Nhận xét:** Ở cả hai thành phố, **mức PM2.5 hiện tại (`pm25_lag_0`) là đặc trưng quan trọng nhất**, phù hợp với tự tương quan cao.
Điểm khác biệt nằm ở các đặc trưng tiếp theo:
- **Hà Nội:** trung bình 6h gần nhất, **lượng mưa 24h** (`rain_24h`) và **mùa** (`season`). Bụi Hà Nội phụ thuộc chế độ mùa và việc mưa rửa trôi.
- **TP.HCM:** **nhiệt độ**, **chu kỳ giờ trong ngày** (`hour_cos`) và **tốc độ gió**. Bụi TP.HCM gắn với nhịp sinh hoạt/giao thông hằng ngày và khả năng phát tán của gió.

Điều này khớp với EDA (Hà Nội có mùa rõ rệt, TP.HCM có đỉnh giờ cao điểm và tương quan mạnh với gió).
Đây cũng là bằng chứng nữa cho thấy nên xây dựng mô hình riêng cho từng thành phố.

### 4.6. Demo hàm suy luận `predict_next`
Nhận lịch sử quan trắc theo giờ đến thời điểm $t$, trả về dự báo PM2.5 lúc $t + 24h$ và mức chất lượng không khí.
Demo: dùng lịch sử kết thúc 24 giờ trước thời điểm có dữ liệu cuối cùng, rồi so với giá trị thực đã biết.

In [ ]:
def predict_next(city, history):
    """history: DataFrame theo giờ (các cột như dữ liệu đã làm sạch), kết thúc tại thời điểm t."""
    model = forecasters[city].models[best_model[city]]
    x_last = build_features(history, city).iloc[[-1]]
    value = float(model.predict(x_last[FEATURES])[0])
    return {'city': city, 'model': best_model[city],
            'forecast_time': x_last.index[0] + pd.Timedelta(hours=HORIZON),
            'pm25_forecast': round(value, 1), 'aqi_level': str(to_aqi_level([value])[0])}


demo_rows = []
for city, df in clean.items():
    t_end = df['pm25'].last_valid_index() - pd.Timedelta(hours=HORIZON)
    history = df.loc[t_end - pd.Timedelta(days=7): t_end]
    res = predict_next(city, history)
    actual = df.loc[res['forecast_time'], 'pm25']
    res.update({'pm25_actual': round(float(actual), 1), 'aqi_actual': str(to_aqi_level([actual])[0])})
    demo_rows.append(res)
display(pd.DataFrame(demo_rows))

### 4.7. Kết luận, hạn chế và hướng phát triển


**Kết luận:**
- Đã xây dựng pipeline dự báo PM2.5 trước 24h, áp dụng giống nhau nhưng **huấn luyện độc lập** cho Hà Nội và TP.HCM, dùng dữ liệu
  quan trắc thật từ OpenAQ kết hợp thời tiết Open-Meteo (12/2024 – 09/2026).
- Trên tập Test, mô hình tốt nhất giảm MAE **13.9% (Hà Nội, RandomForest)** và **10.6% (TP.HCM, Ridge)** so với baseline Persistence,
  dự báo đúng mức AQI 76–83% số giờ.
- Hai thành phố có cơ chế ô nhiễm khác nhau: Hà Nội chi phối bởi mùa và mưa, TP.HCM bởi chu kỳ ngày và gió.

**Hạn chế:**
- Mỗi thành phố chỉ có **một trạm**. Trạm TP.HCM là cảm biến giá rẻ (AirGradient), có thể kém chính xác khi độ ẩm cao.
- Hà Nội thiếu nhiều dữ liệu (32.8%, phần lớn từ cuối 2025), nên tập Test không chứa mùa đông, là mùa ô nhiễm nhất.
- Chỉ có 22 tháng dữ liệu, tức chưa đến 2 chu kỳ mùa, nên mô hình khó học được quy luật theo mùa.
- Mô hình thường dự báo thấp các đợt ô nhiễm cao. Đặc trưng thời tiết dùng giá trị quan trắc tại thời điểm $t$, chưa dùng thời tiết dự báo cho $t+24h$.

**Hướng phát triển:**
- Thêm **dự báo thời tiết** cho thời điểm mục tiêu (Open-Meteo forecast API) và dữ liệu nhiều trạm trong cùng thành phố.
- Dùng hàm mất mát / trọng số mẫu ưu tiên các giờ ô nhiễm cao, hoặc thêm mô hình phân loại "có vượt ngưỡng hay không" để cảnh báo sớm.
- Thử các mô hình chuỗi thời gian chuyên biệt (SARIMAX, LightGBM với nhiều horizon, LSTM) khi có dữ liệu dài hơn.